# Breast Cancer Transcriptomic Analysis

## Exploring Gene Expression Across Breast Cancer Molecular Subtypes

### Objective

Breast cancer is a heterogeneous disease consisting of molecularly
distinct subtypes with different biological characteristics.

This project investigates whether gene-expression profiles can
distinguish breast cancer subtypes using publicly available
transcriptomic data from the NCBI Gene Expression Omnibus (GEO).

Dataset: GSE45827

The processed Series Matrix file was used.

The analysis explores:

- gene-expression patterns across breast cancer subtypes
- dimensionality reduction using PCA
- subtype-associated gene-expression differences
- expression of biologically relevant breast cancer genes
- whether gene-expression profiles can be used to classify tumour subtype

What the design actually involves:

- The study has three broad kinds of material:

* Primary breast tumour samples — tissue taken during surgery before treatment. These tumours are classified into the four subtypes of breast cancer: 41 samples of triple-negative (TN), 30 samples of HER2, 29 samples of Luminal A and 30 samples of Luminal B.
* 11 Normal breast tissue samples — included so expression in cancerous tissue can be compared with non-cancerous breast tissue.
* 14 Breast cancer cell lines — cancer cells maintained and grown in the laboratory, included as another type of breast-cancer model.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import pandas as pd #working with structured/tabular data
import numpy as np #for numerical calculations
import matplotlib.pyplot as plt #for data visualisations

from sklearn.preprocessing import StandardScaler #adjusting different numerical features onto a similiar measuring scale
from sklearn.decomposition import PCA #Principal Component Analysis simplifies/summarises lots of variables so we can see the main patterns

print("Libraries loaded successfully.")

Libraries loaded successfully.


In [2]:
import urllib.request

url = "https://ftp.ncbi.nlm.nih.gov/geo/series/GSE45nnn/GSE45827/matrix/GSE45827_series_matrix.txt.gz"

filename = "GSE45827_series_matrix.txt.gz"

urllib.request.urlretrieve(url, filename)

print("Dataset downloaded successfully.")

Dataset downloaded successfully.


In [3]:
import gzip #lets Python work with the compressed zip file

with gzip.open(filename, "rt") as file:
    for i in range(20):
        print(file.readline().strip()) #only inspects the first 20 lines of the file

!Series_title	"Expression data from Breast cancer subtypes"
!Series_geo_accession	"GSE45827"
!Series_status	"Public on Mar 24 2016"
!Series_submission_date	"Apr 05 2013"
!Series_last_update_date	"Mar 25 2019"
!Series_pubmed_id	"27006338"
!Series_summary	"Expression data from Breast cancer subtypes"
!Series_overall_design	"In a cohort study of primary invasive breast cancer (41 TN, 30 HER2, 29 Luminal A and 30 Luminal B) as well as 11 normal tissues samples and 14 cell lines , we obtained a tumor specimen at surgery before any patient treatment. Total RNA was extracted from all samples and the whole transcriptome was quantified with Affymetrix U133 Plus 2.0 Chips."
!Series_type	"Expression profiling by array"
!Series_contributor	"Tina,,Gruosso"
!Series_contributor	"Yann,,Kieffer"
!Series_contributor	"Thierry,,Dubois"
!Series_contributor	"Fatima,,Mechta-Grigoriou"
!Series_sample_id	"GSM1116084 GSM1116085 GSM1116086 GSM1116087 GSM1116088 GSM1116089 GSM1116090 GSM1116091 GSM1116092 GSM1116

In [4]:
with gzip.open(filename, "rt") as file:
    for line_number, line in enumerate(file):
        if "series_matrix_table_begin" in line:
            print("Expression matrix starts at line:", line_number)
            print(line.strip())
            break #stop at line where the expression matrix starts

Expression matrix starts at line: 67
!series_matrix_table_begin


In [5]:
with gzip.open(filename, "rt") as file:
    lines = file.readlines()

for line in lines[68:73]:
    print(line[:500])

"ID_REF"	"GSM1116084"	"GSM1116085"	"GSM1116086"	"GSM1116087"	"GSM1116088"	"GSM1116089"	"GSM1116090"	"GSM1116091"	"GSM1116092"	"GSM1116093"	"GSM1116094"	"GSM1116095"	"GSM1116096"	"GSM1116097"	"GSM1116098"	"GSM1116099"	"GSM1116100"	"GSM1116101"	"GSM1116102"	"GSM1116103"	"GSM1116104"	"GSM1116105"	"GSM1116106"	"GSM1116107"	"GSM1116108"	"GSM1116109"	"GSM1116110"	"GSM1116111"	"GSM1116112"	"GSM1116113"	"GSM1116114"	"GSM1116115"	"GSM1116116"	"GSM1116117"	"GSM1116118"	"GSM1116119"	"GSM1116120"	"GSM111612
"1007_s_at"	9.47065	9.6744	10.208	10.1142	11.1636	10.0069	9.53932	9.00187	6.54074	10.3551	10.0378	10.6458	10.4495	8.37233	9.48393	10.1414	10.2397	8.60988	9.30953	10.1158	10.2282	10.9337	10.4501	10.774	9.09858	9.82919	9.76393	9.71049	9.63422	9.32985	9.13237	10.2123	10.6347	10.1965	10.2429	9.6089	9.3268	10.5106	10.5697	10.1473	8.34806	9.40561	10.5641	9.73944	11.1704	9.72613	10.9106	8.6077	10.3804	10.3811	9.81215	9.46771	9.67632	11.0246	10.7265	9.75026	10.0241	10.3423	11.1542	11.0368	9.45143	10.43